# 📊 InsightForge AI — MLflow Experiment Tracking & Logging
Tracks all pipeline executions, state parameters, cleaning metrics, chart counts, and insight artifacts in Databricks MLflow.


In [ ]:
%pip install mlflow pandas
dbutils.library.restartPython()


In [ ]:
import mlflow
import os

def log_pipeline_run_to_mlflow(state: dict, experiment_name: str = "InsightForge_Experiments"):
    """
    Logs a full InsightForge pipeline execution to MLflow.
    """
    mlflow.set_experiment(f"/Users/{dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()}/{experiment_name}")
    
    with mlflow.start_run(run_name=f"InsightForge_{state.get('schema_info', {}).get('domain', 'Generic')}"):
        # Log Parameters
        mlflow.log_param("dataset_path", state.get("dataset_path", "unknown"))
        if state.get("raw_df") is not None:
            mlflow.log_param("raw_rows", len(state["raw_df"]))
            mlflow.log_param("raw_cols", len(state["raw_df"].columns))
            
        # Log Metrics
        if state.get("cleaning_report"):
            mlflow.log_metric("nulls_filled", state["cleaning_report"].get("nulls_filled", 0))
            mlflow.log_metric("duplicates_removed", state["cleaning_report"].get("duplicates_removed", 0))
            mlflow.log_metric("outliers_flagged", state["cleaning_report"].get("outliers_flagged", 0))
            
        mlflow.log_metric("charts_count", len(state.get("charts", [])))
        mlflow.log_metric("errors_count", len(state.get("errors", [])))
        
        # Log Artifacts
        if state.get("insights"):
            with open("/tmp/insights_artifact.txt", "w", encoding="utf-8") as f:
                f.write(state["insights"])
            mlflow.log_artifact("/tmp/insights_artifact.txt")
            
        print("✅ Run successfully logged to MLflow!")
